In [19]:
import re
import json
import os
from datetime import datetime, timedelta

class AgentePersonalAvanzado:
    def __init__(self, archivo="actividades.json"):
        self.archivo = archivo
        self.actividades = []  # lista de dicts: nombre, inicio, duracion, prioridad, recurrencia, tipo
        self.cargar()

    def cargar(self):
        if os.path.exists(self.archivo):
            with open(self.archivo, "r") as f:
                try:
                    self.actividades = json.load(f)
                except json.JSONDecodeError: # Handle empty or malformed JSON file
                    self.actividades = []

    def guardar(self):
        with open(self.archivo, "w") as f:
            json.dump(self.actividades, f, indent=2)

    def limpiar_actividades(self):
        self.actividades = []
        if os.path.exists(self.archivo):
            os.remove(self.archivo)
        self.guardar() # Recreate an empty file
        return "Todas las actividades han sido eliminadas."

    def eliminar_actividad(self, nombre_actividad):
        nombre_actividad_lower = nombre_actividad.lower()
        actividades_antes = len(self.actividades)
        self.actividades = [act for act in self.actividades if act['nombre'].lower() != nombre_actividad_lower]
        if len(self.actividades) < actividades_antes:
            self.guardar()
            return f"🗑️ Actividad(es) '{nombre_actividad}' eliminada(s)."
        else:
            return f"❌ No se encontró la actividad '{nombre_actividad}'."

    # --- Interpretación de lenguaje natural ---
    def extraer_fecha_hora(self, texto):
        # busca patrones como "mañana a las 15:30", "hoy 10am", "2026-06-15 14:00"
        ahora = datetime.now()
        if "mañana" in texto:
            base = ahora + timedelta(days=1)
        elif "pasado mañana" in texto:
            base = ahora + timedelta(days=2)
        elif "hoy" in texto:
            base = ahora
        else:
            # busca fecha explícita YYYY-MM-DD
            match = re.search(r"(\d{4}-\d{2}-\d{2})", texto)
            if match:
                base = datetime.strptime(match.group(1), "%Y-%m-%d")
            else:
                base = ahora  # por defecto hoy
        # buscar hora
        hora_match = re.search(r"(\d{1,2}):(\d{2})", texto)
        if not hora_match:
            hora_match = re.search(r"(\d{1,2})\s*(am|pm)", texto, re.IGNORECASE)
            if hora_match:
                h = int(hora_match.group(1))
                if hora_match.group(2).lower() == "pm" and h != 12:
                    h += 12
                return base.replace(hour=h, minute=0)
        if hora_match:
            h = int(hora_match.group(1))
            m = int(hora_match.group(2)) if hora_match.lastindex >= 2 else 0
            return base.replace(hour=h, minute=m)
        # si no hay hora, asumir 09:00
        return base.replace(hour=9, minute=0)

    def extraer_duracion(self, texto):
        match = re.search(r"(\d+(?:\.\d+)?)\s*(h|horas|min)", texto.lower())
        if match:
            valor = float(match.group(1))
            if match.group(2).startswith("min"):
                return valor / 60.0
            return valor
        return 1.0  # 1 hora por defecto

    def extraer_prioridad(self, texto):
        if "alta" in texto:
            return 1
        elif "baja" in texto:
            return 3
        return 2

    def extraer_recurrencia(self, texto):
        if "diario" in texto or "cada día" in texto:
            return "diaria"
        elif "semanal" in texto or "cada semana" in texto:
            return "semanal"
        return "ninguna"

    def extraer_tipo(self, texto):
        if "estudio" in texto or "clase" in texto:
            return "estudio"
        elif "gym" in texto or "deporte" in texto:
            return "deporte"
        elif "descanso" in texto or "relaj" in texto:
            return "descanso"
        else:
            return "otro"

    # --- Agregar actividad desde lenguaje natural ---
    def agregar_actividad(self, comando):
        # Refined regex to capture activity name more broadly, stopping before known keywords
        nombre_match = re.search(r"(?:agregar|nueva)\s+actividad\s+([\w\sáéíóúñ\-]+?)(?:\s+para|\s+a las|\s+mañana|\s+hoy|\s+con prioridad|\s+duracion|$)", comando)
        if nombre_match:
            nombre = nombre_match.group(1).strip()
        else:
            # Fallback if regex fails, tries to take words after 'agregar actividad'
            words = comando.split()
            if len(words) > 2 and words[0] in ['agregar', 'nueva'] and words[1] == 'actividad':
                nombre_parts = []
                skip_keywords = ['para', 'a las', 'mañana', 'hoy', 'con', 'prioridad', 'duracion'] # More explicit keywords
                for word_idx in range(2, len(words)):
                    current_word = words[word_idx].lower()
                    if any(keyword in current_word for keyword in skip_keywords):
                        break
                    nombre_parts.append(words[word_idx])
                nombre = ' '.join(nombre_parts).strip()
                if not nombre: # If still empty, use a default
                    nombre = 'actividad'
            else:
                nombre = "actividad" # Default if no clear name can be parsed

        inicio = self.extraer_fecha_hora(comando)
        duracion = self.extraer_duracion(comando)
        prioridad = self.extraer_prioridad(comando)
        recurrencia = self.extraer_recurrencia(comando)
        tipo = self.extraer_tipo(comando)

        nueva_inicio = inicio
        nueva_fin = nueva_inicio + timedelta(hours=duracion)

        conflictos_encontrados = self.conflictos(nueva_inicio, nueva_fin)
        warning_msg = ""
        if conflictos_encontrados:
            conflictos_str = ", ".join(conflictos_encontrados)
            warning_msg = f"⚠️ ¡CUIDADO! Esta actividad se solapa con: {conflictos_str}."

        actividad = {
            "nombre": nombre,
            "inicio": inicio.isoformat(),
            "duracion": duracion,
            "prioridad": prioridad,
            "recurrencia": recurrencia,
            "tipo": tipo
        }
        self.actividades.append(actividad)
        self.guardar()
        return_message = f"✅ Actividad '{nombre}' agregada el {inicio.strftime('%Y-%m-%d %H:%M')} por {duracion}h (prioridad {prioridad})"
        if warning_msg:
            return_message += f"\n{warning_msg}"
        return return_message

    # --- Detectar conflictos ---
    def conflictos(self, nueva_inicio, nueva_fin, omitir_id=None):
        conflictos = []
        for i, act in enumerate(self.actividades):
            if omitir_id is not None and i == omitir_id:
                continue
            a_inicio = datetime.fromisoformat(act["inicio"])
            a_fin = a_inicio + timedelta(hours=act["duracion"])
            if max(nueva_inicio, a_inicio) < min(nueva_fin, a_fin):
                conflictos.append(act["nombre"])
        return conflictos

    # --- Generar horario (ordenado por inicio) ---
    def generar_horario(self):
        # expandir recurrencias en un rango de 7 días
        hoy = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        fin_semana = hoy + timedelta(days=7)
        eventos = []
        for act in self.actividades:
            inicio = datetime.fromisoformat(act["inicio"])
            if act["recurrencia"] == "diaria":
                fecha = inicio
                while fecha <= fin_semana:
                    eventos.append((fecha, act))
                    fecha += timedelta(days=1)
            elif act["recurrencia"] == "semanal":
                fecha = inicio
                while fecha <= fin_semana:
                    eventos.append((fecha, act))
                    fecha += timedelta(days=7)
            else:
                eventos.append((inicio, act))
        eventos.sort(key=lambda x: x[0])
        horario = []
        for fecha, act in eventos:
            fin = fecha + timedelta(hours=act["duracion"])
            horario.append({
                "actividad": act["nombre"],
                "inicio": fecha.strftime("%Y-%m-%d %H:%M"),
                "fin": fin.strftime("%Y-%m-%d %H:%M"),
                "tipo": act["tipo"]
            })
        return horario

    def mostrar_horario(self, horario):
        if not horario:
            print("📭 No hay actividades programadas.")
            return
        print("\n🗓️  HORARIO SEMANAL:")
        for bloque in horario:
            print(f"{bloque['inicio']} → {bloque['fin']} | {bloque['actividad']} ({bloque['tipo']})")

    # --- Punto de entrada principal ---
    def procesar(self, comando):
        comando = comando.lower()
        if "agregar" in comando or "nueva actividad" in comando:
            return self.agregar_actividad(comando)
        elif "horario" in comando or "mostrar" in comando:
            self.mostrar_horario(self.generar_horario())
            return "Horario generado."
        elif "conflictos" in comando:
            # The conflict detection is now integrated into agregar_actividad
            return "La detección de conflictos se realiza automáticamente al agregar una actividad."
        elif "limpiar" in comando or "borrar todo" in comando:
            return self.limpiar_actividades()
        elif "eliminar" in comando or "borrar actividad" in comando:
            nombre_match = re.search(r"(?:eliminar|borrar)\s+actividad\s+([\w\sáéíóúñ\-]+)", comando)
            if nombre_match:
                nombre_a_eliminar = nombre_match.group(1).strip()
                return self.eliminar_actividad(nombre_a_eliminar)
            else:
                return "Por favor, especifica el nombre de la actividad a eliminar (ej: 'eliminar actividad estudiar python')."
        else:
            return "Comandos soportados: 'agregar actividad <nombre> para mañana a las 10am duración 2h prioridad alta', 'mostrar horario', 'limpiar actividades', 'eliminar actividad <nombre>'. La detección de conflictos es automática."


## Configuración de la API con Flask

Primero, necesitamos instalar Flask, un microframework para Python que nos ayudará a crear la API.

In [22]:
# Instalar Flask
!pip install Flask

Ahora, crearemos una aplicación Flask y definiremos los *endpoints* (rutas URL) que permitirán a otras aplicaciones comunicarse con nuestro `AgentePersonalAvanzado`.

In [23]:
from flask import Flask, request, jsonify

app = Flask(__name__)
agente_api = AgentePersonalAvanzado() # Instancia de nuestro agente

@app.route('/')
def home():
    return '¡Bienvenido a la API de Agente Personal! Usa los endpoints como /agregar_actividad, /mostrar_horario, etc.'

@app.route('/agregar_actividad', methods=['POST'])
def agregar_actividad_api():
    data = request.get_json()
    if not data or 'comando' not in data:
        return jsonify({'error': 'Se requiere un comando para agregar actividad'}), 400

    comando = data['comando']
    resultado = agente_api.procesar(comando)
    return jsonify({'mensaje': resultado})

@app.route('/mostrar_horario', methods=['GET'])
def mostrar_horario_api():
    horario = agente_api.generar_horario()
    return jsonify({'horario': horario})

@app.route('/limpiar_actividades', methods=['POST'])
def limpiar_actividades_api():
    resultado = agente_api.limpiar_actividades()
    return jsonify({'mensaje': resultado})

@app.route('/eliminar_actividad', methods=['POST'])
def eliminar_actividad_api():
    data = request.get_json()
    if not data or 'nombre' not in data:
        return jsonify({'error': 'Se requiere el nombre de la actividad a eliminar'}), 400

    nombre_actividad = data['nombre']
    resultado = agente_api.procesar(f"eliminar actividad {nombre_actividad}")
    return jsonify({'mensaje': resultado})

Para ejecutar la API dentro de Colab, usaremos `app.run()`. Ten en cuenta que si cierras esta pestaña de Colab, la API dejará de ejecutarse. Si quisieras un despliegue persistente, necesitarías un entorno de servidor dedicado.

Una vez que la celda se ejecute, verás una URL (generalmente `http://127.0.0.1:5000/`) y, si usas `ngrok` o una herramienta similar para exponer tu puerto local, podrás acceder a ella desde fuera de Colab. En este ejemplo, simplemente se ejecutará dentro del entorno de Colab.

In [ ]:
# Ejecutar la aplicación Flask
# Esto bloqueará la celda hasta que detengas la ejecución.
# En un entorno de producción, usarías un servidor WSGI como Gunicorn.
app.run(debug=True, host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


### Cómo probar la API (ejemplos)

Una vez que la celda anterior esté ejecutándose (y muestre 'Running on http://0.0.0.0:5000/'), puedes probar los endpoints usando `curl` en una nueva celda de código, o con bibliotecas como `requests` en Python, o herramientas como Postman/Insomnia.

**Ejemplo: Agregar una actividad** (POST a `/agregar_actividad`)

```python
import requests

url = 'http://127.0.0.1:5000/agregar_actividad'
data = {'comando': 'agregar actividad Estudiar Flask para mañana a las 14:00 duración 2h prioridad alta'}
response = requests.post(url, json=data)
print(response.json())
```

**Ejemplo: Mostrar el horario** (GET a `/mostrar_horario`)

```python
import requests

url = 'http://127.0.0.1:5000/mostrar_horario'
response = requests.get(url)
print(response.json())
```

**Ejemplo: Eliminar una actividad** (POST a `/eliminar_actividad`)

```python
import requests

url = 'http://127.0.0.1:5000/eliminar_actividad'
data = {'nombre': 'Estudiar Flask'}
response = requests.post(url, json=data)
print(response.json())
```

**Ejemplo: Limpiar actividades** (POST a `/limpiar_actividades`)

```python
import requests

url = 'http://127.0.0.1:5000/limpiar_actividades'
response = requests.post(url)
print(response.json())
```
